# Task 1.2: Key Assumptions
**Paper**: *Ensemble of Exemplar-SVMs for Object Detection and Beyond* — Malisiewicz, Gupta & Efros (ICCV 2011)

---

## Assumption 1: Each Exemplar is Visually Distinctive

**Assumption**: Each positive exemplar is sufficiently distinctive in the HOG feature space to define a meaningful linear decision boundary when used as the sole positive example against a large set of negatives.

**Why the method needs it**: The entire Exemplar-SVM framework rests on training one SVM per positive exemplar. If an exemplar's HOG representation is not distinctive — that is, it is very similar to many negative windows — then the linear SVM trained on it cannot find a hyperplane that meaningfully separates it from the negatives. The resulting classifier would fire indiscriminately, producing many false positives, and the exemplar's contribution to the ensemble would be noise rather than signal. The paper's claim that each Exemplar-SVM 'will produce a set of positive firings that exhibit a strong visual resemblance to the exemplar' (Section 4) requires that the exemplar's appearance is distinct enough from the background to learn a discriminative boundary.

**Violation scenario**: Consider an urban scene dataset where many objects (utility poles, lamp posts, bollards) share the same vertical-stick HOG pattern. An exemplar of a 'person standing' that has a similar vertical-stick signature would produce an Exemplar-SVM that fires on all vertical objects. In real-world settings, this can also happen for heavily occluded or truncated exemplars where only a small non-distinctive fragment is visible — the SVM trained on such an exemplar would match too many background windows.

**Paper reference**: Section 3, Figure 2 — the per-exemplar training procedure implicitly assumes the positive is separable; Section 4 — *'exemplar-SVM … will produce a set of positive firings that exhibit a strong visual resemblance to the exemplar'*.

---

## Assumption 2: Calibration Scores are Reliable Proxies for Inter-Exemplar Quality

**Assumption**: The Platt scaling sigmoid fitted per-exemplar on a held-out validation set produces calibrated scores that are reliable and comparable across all exemplars, enabling meaningful score-based ranking and non-maximum suppression.

**Why the method needs it**: Since each Exemplar-SVM is trained independently with its own positive, the raw SVM decision scores have different scales and distributions. The ensemble relies on comparing calibrated scores from different exemplars to decide which detection to keep during non-maximum suppression (Section 4.1). If the calibration is unreliable — for instance, if the validation set does not adequately represent the test distribution — then the fitted sigmoid may produce miscalibrated probabilities, causing a poor-quality exemplar to dominate over a better one. The paper explicitly states: *'different exemplars will offer drastically different generalization potential'* (Section 3.1), making calibration essential for fair inter-exemplar comparison.

**Violation scenario**: If the validation set comes from a domain significantly different from the test set (e.g., validation images are from indoor scenes but test images are outdoor street views), the Platt scaling parameters estimated on validation data would not generalise. The sigmoid curve might compress all scores into a narrow range or invert the ranking of good vs. bad exemplars. Similarly, if the validation set is very small, the two sigmoid parameters (α, β) could overfit, producing unreliable calibration.

**Paper reference**: Section 3.1 — *'we fit a logistic function to these scores. The two scalar parameters of this function, alpha and beta, are estimated per-exemplar via maximum likelihood'*; the calibration equation $f(x | w_E, \alpha_E, \beta_E)$.

---

## Assumption 3: Linear Separability in the HOG Feature Space

**Assumption**: Each positive exemplar is approximately linearly separable from the negative pool in the HOG feature space — that is, there exists a hyperplane that can separate the single positive exemplar from the negatives with reasonable accuracy.

**Why the method needs it**: The paper uses **linear** SVMs exclusively, not kernel SVMs. This is a deliberate choice for computational efficiency — the authors need to train thousands of SVMs (one per exemplar), and linear SVMs are orders of magnitude faster than kernel SVMs for high-dimensional HOG features. However, this means the method can only learn linear decision boundaries. If the relationship between the exemplar and negatives is highly nonlinear — for example, if the exemplar is similar to negatives in some feature dimensions but different in others in a nonlinear way — the linear SVM will fail to separate them. The success of the method depends on HOG features creating a space where linear separation is approximately achieved.

**Violation scenario**: Artistic or stylised objects (cartoons, abstract art, paintings) may have HOG representations that interleave with real-object negative windows in complex nonlinear patterns because artistic renderings distort the local edge statistics that HOG captures. In this setting, a linear SVM trained on one artistic exemplar would fail to separate it from real-world negatives, and a kernel SVM or deeper feature representation would be needed.

**Paper reference**: Section 3 — *'a separate linear SVM classifier for each exemplar'*; the hinge-loss objective $\Omega_E$ uses the linear scoring function $w^T x + b$ with no kernel mapping. Section 5 discusses Exemplar-LDA as an efficient alternative that also assumes linearity.